In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd '/content/drive/MyDrive/Syncpol'

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
# @title
import pyarrow.ipc as ipc
import pandas as pd
import os
import gc # Import garbage collector

filename = '/content/drive/MyDrive/Syncpol/Data/cleaned_data_final/EP_corpus_sentences_cleaned_Translated.feather'
output_dir = '/content/drive/MyDrive/Syncpol/Ep data chunks'

# Create output directory if it doesn't exist
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"Created output directory: {output_dir}")

# Open the feather file for reading
reader = ipc.open_file(filename)

desired_batch_size = 1000000
current_batch_data = []
current_batch_row_count = 0
chunk_index = 0

# Iterate through internal record batches
for i in range(reader.num_record_batches):
    batch = reader.get_batch(i)
    df_internal_batch = batch.to_pandas()

    current_batch_data.append(df_internal_batch)
    current_batch_row_count += len(df_internal_batch)

    # If accumulating data reaches or exceeds the desired batch size, process it
    if current_batch_row_count >= desired_batch_size:
        # Concatenate the accumulated internal batches into one larger chunk
        df_chunk = pd.concat(current_batch_data, ignore_index=True)

        print(f"Processing and saving chunk {chunk_index} with shape:", df_chunk.shape)

        # Define output path for the chunk
        output_path = os.path.join(output_dir, f'ep_chunk_{chunk_index}.feather')

        # Save the chunk to a feather file
        df_chunk.to_feather(output_path)
        print(f"Saved chunk {chunk_index} to {output_path}")

        # Explicitly delete the DataFrame chunk and run garbage collector
        del df_chunk
        gc.collect()

        # Reset for the next chunk
        current_batch_data = []
        current_batch_row_count = 0
        chunk_index += 1

# Process any remaining data in the last partial chunk
if current_batch_data:
    df_chunk = pd.concat(current_batch_data, ignore_index=True)
    print(f"Processing and saving final chunk {chunk_index} with shape:", df_chunk.shape)
    output_path = os.path.join(output_dir, f'ep_chunk_{chunk_index}.feather')
    df_chunk.to_feather(output_path)
    print(f"Saved final chunk {chunk_index} to {output_path}")

    # Explicitly delete the DataFrame chunk and run garbage collector
    del df_chunk
    gc.collect()

print("Finished processing and saving all chunks.")

Processing and saving chunk 0 with shape: (1048576, 18)
Saved chunk 0 to /content/drive/MyDrive/Syncpol/Ep data chunks/ep_chunk_0.feather
Processing and saving chunk 1 with shape: (1062010, 18)
Saved chunk 1 to /content/drive/MyDrive/Syncpol/Ep data chunks/ep_chunk_1.feather
Processing and saving chunk 2 with shape: (1009164, 18)
Saved chunk 2 to /content/drive/MyDrive/Syncpol/Ep data chunks/ep_chunk_2.feather
Processing and saving chunk 3 with shape: (1039873, 18)
Saved chunk 3 to /content/drive/MyDrive/Syncpol/Ep data chunks/ep_chunk_3.feather
Processing and saving final chunk 4 with shape: (337745, 18)
Saved final chunk 4 to /content/drive/MyDrive/Syncpol/Ep data chunks/ep_chunk_4.feather
Finished processing and saving all chunks.


In [ ]:
ep = ep.rename(columns={'current': 'sentence'})


NameError: name 'ep' is not defined

In [ ]:
# --------------------------
#  Tokenization
# --------------------------
from transformers import AutoTokenizer # Import AutoTokenizer here
from datasets import Dataset # Import Dataset from the datasets library

model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(batch["sentence"], padding="max_length", truncation=True, max_length=256)

# Tokenize all datasets
print("Tokenizing datasets...")
# Convert the pandas DataFrame to a Hugging Face Dataset
ep_dataset = Dataset.from_pandas(ep)
ep_dataset = ep_dataset.map(tokenize, batched=True)
ep_dataset = ep_dataset.remove_columns(["sentence"])

KeyboardInterrupt: 

In [ ]:
#*********************************
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import os
from tqdm.auto import tqdm
import pyarrow.feather as feather
import pandas as pd # Import pandas
from datasets import Dataset # Import Dataset here
from torch.utils.data import DataLoader
import gc

model_dir = '/content/drive/MyDrive/Climate Mitigation Classification/final_model_silver_pretrained_gold_finetuned'
model_name = "roberta-base"

# Load the model
if os.path.isdir(model_dir):
    # Use GPU if available
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    model = AutoModelForSequenceClassification.from_pretrained(model_dir)
    model.to(device)
    model.eval()  # Set to evaluation mode

    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # Define tokenization function
    def tokenize(examples):
        return tokenizer(
            examples["sentence"],
            padding="max_length",
            truncation=True,
            max_length=512 # Keep max_length consistent or adjust as needed
        )

    # Input and output directories
    input_dir = '/content/drive/MyDrive/Syncpol/Ep data chunks'
    output_dir = '/content/drive/MyDrive/Syncpol/EP climate classified'

    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    print(f"Created output directory: {output_dir}")

    # Get list of chunk files
    chunk_files = sorted([f for f in os.listdir(input_dir) if f.startswith('ep_chunk_') and f.endswith('.feather')])

    # Batch size for DataLoader (adjust if OOM)
    batch_size = 128 # Reduced batch size

    print(f"Found {len(chunk_files)} chunks in {input_dir}. Processing...")

    # Iterate through each chunk file
    for chunk_file in tqdm(chunk_files, desc="Processing Chunk Files"):
        chunk_path = os.path.join(input_dir, chunk_file)
        chunk_output_path = os.path.join(output_dir, f'classified_{chunk_file}') # Define output path for classified chunk

        # Skip if classified chunk already exists (for resuming)
        if os.path.exists(chunk_output_path):
            print(f"Skipping {chunk_file}, classified output already exists.")
            continue

        print(f"\nLoading and processing {chunk_file} in smaller chunks...")

        # Load the chunk using pandas read_feather
        try:
            df_chunk = pd.read_feather(chunk_path)
        except Exception as e:
            print(f"Error loading chunk {chunk_file}: {e}")
            continue # Skip to next chunk if loading fails

        # Ensure 'sentence' column exists (rename if needed, assuming 'current' was renamed to 'sentence' earlier)
        if 'current' in df_chunk.columns and 'sentence' not in df_chunk.columns:
             df_chunk = df_chunk.rename(columns={'current': 'sentence'})
        elif 'sentence' not in df_chunk.columns:
             print(f"Warning: 'sentence' column not found in {chunk_file}. Skipping.")
             del df_chunk # Clear memory
             gc.collect()
             continue # Skip to next chunk

        # Process the loaded chunk in smaller sub-chunks
        sub_chunk_size = 300000  # Define smaller sub-chunk size
        total_rows_in_chunk = len(df_chunk)
        sub_chunk_predictions = []

        for i in tqdm(range(0, total_rows_in_chunk, sub_chunk_size), desc=f"Processing sub-chunks of {chunk_file}", leave=False):
            sub_df = df_chunk.iloc[i : i + sub_chunk_size].copy() # Use .copy() to avoid SettingWithCopyWarning

            # Convert sub-chunk pandas DataFrame to Hugging Face Dataset
            sub_dataset = Dataset.from_pandas(sub_df)
            del sub_df # Clear pandas DataFrame from memory
            gc.collect()

            # Tokenize only the 'sentence' column
            sub_dataset = sub_dataset.map(tokenize, batched=True, batch_size=512) # Reduced map batch size
            # Prepare columns for model input - only keep input_ids and attention_mask
            columns_to_set_format = ['input_ids', 'attention_mask']
            sub_dataset.set_format(type='torch', columns=columns_to_set_format)

            # Use DataLoader for efficient batching for model inference
            dataloader = DataLoader(sub_dataset, batch_size=batch_size, shuffle=False)

            # Process with GPU/CPU
            model.eval() # Ensure model is in evaluation mode
            with torch.no_grad():
                for batch in tqdm(dataloader, desc=f"Classifying sub-chunk", leave=False):
                    # Move batch to device
                    input_ids = batch['input_ids'].to(device)
                    attention_mask = batch['attention_mask'].to(device)

                    # Forward pass
                    outputs = model(input_ids, attention_mask=attention_mask)

                    # Get predictions and move back to CPU immediately
                    predictions = torch.argmax(outputs.logits, dim=1).cpu().numpy()
                    sub_chunk_predictions.extend(predictions)

            # Clear memory after each sub-chunk
            del sub_dataset, dataloader
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()


        # Add predictions to the original df_chunk before saving
        df_chunk['predicted_class'] = sub_chunk_predictions

        # Save the classified chunk
        try:
            feather.write_feather(df_chunk, chunk_output_path)
            print(f"Saved classified chunk to {chunk_output_path}")
        except Exception as e:
            print(f"Error saving classified chunk {chunk_file}: {e}")


        # Clear memory after each main chunk
        del df_chunk, sub_chunk_predictions
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


    print("\nFinished processing all chunks.")

else:
    print(f"Error: Model directory not found at {model_dir}")
    print("Please ensure the 'model_dir' variable points to a valid directory containing the model files.")

Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

...

Processing Chunk Files:   0%|          | 0/5 [00:00<?, ?it/s]

Skipping ep_chunk_0.feather, classified output already exists.
Skipping ep_chunk_1.feather, classified output already exists.
Skipping ep_chunk_2.feather, classified output already exists.
Skipping ep_chunk_3.feather, classified output already exists.

Loading and processing ep_chunk_4.feather in smaller chunks...


Processing sub-chunks of ep_chunk_4.feather:   0%|          | 0/2 [00:00<?, ?it/s]

Map:   0%|          | 0/300000 [00:00<?, ? examples/s]

Classifying sub-chunk:   0%|          | 0/2344 [00:00<?, ?it/s]

In [ ]:
# @title
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import os
from tqdm.auto import tqdm
import pyarrow.feather as feather
from datasets import Dataset # Import Dataset here
from torch.utils.data import DataLoader
import gc

model_dir = 'final_model_silver_pretrained_gold_finetuned'
model_name = "roberta-base"

# Load the model
if os.path.isdir(model_dir):
    # 1. Use GPU if available
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    model = AutoModelForSequenceClassification.from_pretrained(model_dir)
    model.to(device)
    model.eval()  # Set to evaluation mode

    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # Define tokenization function (added here)
    def tokenize(examples):
        return tokenizer(
            examples["sentence"],
            padding="max_length",
            truncation=True,
            max_length=512
        )

    # 2. Use larger chunks and batch size
    chunk_size = 500000  # Increase chunk size (adjust if OOM)
    batch_size = 256  # Large batch size for GPU (adjust if OOM)

    # Get the total number of rows
    total_rows = len(com)
    num_chunks = (total_rows + chunk_size - 1) // chunk_size

    print(f"Processing {total_rows} rows in {num_chunks} chunks with batch size {batch_size}...")

    all_predictions = []

    # Create checkpoint directory
    checkpoint_dir = 'inference_checkpoints'
    os.makedirs(checkpoint_dir, exist_ok=True)

    # Check if there are existing checkpoints to resume from
    existing_checkpoints = sorted([f for f in os.listdir(checkpoint_dir) if f.startswith('chunk_')])
    start_chunk = 0

    if existing_checkpoints:
        last_checkpoint = existing_checkpoints[-1]
        start_chunk = int(last_checkpoint.split('_')[1].split('.')[0]) + 1
        print(f"Resuming from chunk {start_chunk}...")

        # Load existing predictions
        import pickle
        for checkpoint_file in existing_checkpoints:
            with open(os.path.join(checkpoint_dir, checkpoint_file), 'rb') as f:
                chunk_predictions = pickle.load(f)
                all_predictions.extend(chunk_predictions)

    # Iterate through the DataFrame in chunks
    for chunk_idx, i in enumerate(tqdm(range(start_chunk * chunk_size, total_rows, chunk_size),
                                        initial=start_chunk,
                                        total=num_chunks,
                                        desc="Processing Chunks")):
        # Define chunk_number within the loop
        chunk_number = start_chunk + chunk_idx
        chunk_df = com.iloc[i : i + chunk_size]

        # Convert to Dataset and tokenize
        chunk_dataset = Dataset.from_pandas(chunk_df)
        chunk_dataset = chunk_dataset.map(tokenize, batched=True, batch_size=1000)
        chunk_dataset = chunk_dataset.remove_columns(["sentence"])

        # Set format for PyTorch
        chunk_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask'])

        # Use DataLoader for efficient batching
        dataloader = DataLoader(chunk_dataset, batch_size=batch_size, shuffle=False)

        # Store chunk predictions separately
        chunk_predictions = []

        # Process with GPU
        with torch.no_grad():
            for batch in tqdm(dataloader,
                            desc=f"Chunk {chunk_number + 1}/{num_chunks}", # Corrected chunk number display
                            leave=False):
                # Move batch to device
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)

                # Forward pass
                outputs = model(input_ids, attention_mask=attention_mask)

                # Get predictions and move back to CPU immediately
                predictions = torch.argmax(outputs.logits, dim=1).cpu().numpy()
                chunk_predictions.extend(predictions)

        # Add chunk predictions to all predictions
        all_predictions.extend(chunk_predictions)

        # Save checkpoint after each chunk
        checkpoint_file = os.path.join(checkpoint_dir, f'chunk_{chunk_number}.pkl')
        import pickle
        with open(checkpoint_file, 'wb') as f:
            pickle.dump(chunk_predictions, f)
        print(f"Saved checkpoint: {checkpoint_file}")

        # Clear memory after each chunk
        del chunk_df, chunk_dataset, dataloader, chunk_predictions
        gc.collect()
        torch.cuda.empty_cache()

    # Add predictions to the DataFrame
    com['predicted_class'] = all_predictions

    # Save final results
    output_file = 'com_with_predictions.feather'
    feather.write_feather(com, output_file)
    print(f"\nFinal results saved to: {output_file}")

    # Optional: Clean up checkpoint files after successful completion
    import shutil
    shutil.rmtree(checkpoint_dir)
    print(f"Checkpoint directory '{checkpoint_dir}' removed after successful completion.")

    display(com.head())

else:
    print(f"Error: Model directory not found at {model_dir}")
    print("Please ensure the 'model_dir' variable points to a valid directory containing the model files.")

In [ ]:
# @title
import pandas as pd
com=pd.read_feather('/content/drive/MyDrive/Syncpol/Ep data chunks/ep_chunk_0.feather')

In [ ]:
# @title
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import os
from tqdm.auto import tqdm
import pyarrow.feather as feather
from datasets import Dataset # Import Dataset here
from torch.utils.data import DataLoader
import gc

model_dir='/content/drive/MyDrive/Climate Mitigation Classification/final_model_silver_pretrained_gold_finetuned'
model_name = "roberta-base"

# Load the model
if os.path.isdir(model_dir):
    # 1. Use GPU if available
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    model = AutoModelForSequenceClassification.from_pretrained(model_dir)
    model.to(device)
    model.eval()  # Set to evaluation mode

    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # Define tokenization function (added here)
    def tokenize(examples):
        return tokenizer(
            examples["sentence"],
            padding="max_length",
            truncation=True,
            max_length=512
        )

    # 2. Use larger chunks and batch size
    chunk_size = 100000  # Increase chunk size (adjust if OOM)
    batch_size = 256  # Large batch size for GPU (adjust if OOM)

    # Get the total number of rows
    total_rows = len(com)
    num_chunks = (total_rows + chunk_size - 1) // chunk_size

    print(f"Processing {total_rows} rows in {num_chunks} chunks with batch size {batch_size}...")

    all_predictions = []

    # Create checkpoint directory
    checkpoint_dir = 'inference_checkpoints'
    os.makedirs(checkpoint_dir, exist_ok=True)

    # Check if there are existing checkpoints to resume from
    existing_checkpoints = sorted([f for f in os.listdir(checkpoint_dir) if f.startswith('chunk_')])
    start_chunk = 0

    if existing_checkpoints:
        last_checkpoint = existing_checkpoints[-1]
        start_chunk = int(last_checkpoint.split('_')[1].split('.')[0]) + 1
        print(f"Resuming from chunk {start_chunk}...")

        # Load existing predictions
        import pickle
        for checkpoint_file in existing_checkpoints:
            with open(os.path.join(checkpoint_dir, checkpoint_file), 'rb') as f:
                chunk_predictions = pickle.load(f)
                all_predictions.extend(chunk_predictions)

    # Iterate through the DataFrame in chunks
    for chunk_idx, i in enumerate(tqdm(range(start_chunk * chunk_size, total_rows, chunk_size),
                                        initial=start_chunk,
                                        total=num_chunks,
                                        desc="Processing Chunks")):
        # Define chunk_number within the loop
        chunk_number = start_chunk + chunk_idx
        chunk_df = com.iloc[i : i + chunk_size]

        # Convert to Dataset and tokenize
        chunk_dataset = Dataset.from_pandas(chunk_df)
        chunk_dataset = chunk_dataset.map(tokenize, batched=True, batch_size=1000)
        chunk_dataset = chunk_dataset.remove_columns(["sentence"])

        # Set format for PyTorch
        chunk_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask'])

        # Use DataLoader for efficient batching
        dataloader = DataLoader(chunk_dataset, batch_size=batch_size, shuffle=False)

        # Store chunk predictions separately
        chunk_predictions = []

        # Process with GPU
        with torch.no_grad():
            for batch in tqdm(dataloader,
                            desc=f"Chunk {chunk_number + 1}/{num_chunks}", # Corrected chunk number display
                            leave=False):
                # Move batch to device
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)

                # Forward pass
                outputs = model(input_ids, attention_mask=attention_mask)

                # Get predictions and move back to CPU immediately
                predictions = torch.argmax(outputs.logits, dim=1).cpu().numpy()
                chunk_predictions.extend(predictions)

        # Add chunk predictions to all predictions
        all_predictions.extend(chunk_predictions)

        # Save checkpoint after each chunk
        checkpoint_file = os.path.join(checkpoint_dir, f'chunk_{chunk_number}.pkl')
        import pickle
        with open(checkpoint_file, 'wb') as f:
            pickle.dump(chunk_predictions, f)
        print(f"Saved checkpoint: {checkpoint_file}")

        # Clear memory after each chunk
        del chunk_df, chunk_dataset, dataloader, chunk_predictions
        gc.collect()
        torch.cuda.empty_cache()

    # Add predictions to the DataFrame
    com['predicted_class'] = all_predictions

    # Save final results
    output_file = 'ep_with_predictions.feather'
    feather.write_feather(com, output_file)
    print(f"\nFinal results saved to: {output_file}")

    # Optional: Clean up checkpoint files after successful completion
    import shutil
    shutil.rmtree(checkpoint_dir)
    print(f"Checkpoint directory '{checkpoint_dir}' removed after successful completion.")

    display(com.head())

else:
    print(f"Error: Model directory not found at {model_dir}")
    print("Please ensure the 'model_dir' variable points to a valid directory containing the model files.")

Using device: cpu


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


NameError: name 'com' is not defined